# **Notebook Overview: Sentinel V3**

### **TL;DR**

I created this notebook to fine-tune and test the Llama 3.2 model specifically for pharmacovigilance. Since standard models struggle with "social media medicalese," I am using this notebook to teach the model using expert-labeled data (CADEC) and then deploying it to hunt for adverse drug reactions in my own raw scraped data from Reddit. This is purely about model behavior and inference on specific datasets.

### **Process Flow**

**1. Environment Setup**
Installs the specific dependencies required for LoRA fine-tuning and efficient 4-bit loading of the Llama 3.2 model.

**2. Data Loading & Formatting**
Reads the two core datasets. It prepares the expert-annotated data (CADEC) for training/validation and loads the raw social media data (Reddit) for the final detection run.

**3. Model Initialization**
Loads the Llama 3.2 model with quantization (memory optimization) to fit on the available GPU, attaching the tokenizer that handles the specific chat templates.

**4. Fine-Tuning / Instruction Logic**
Uses the annotated CADEC data to show the model exactly how to extract entities (Drugs, Symptoms) from informal text. (Depending on the specific cell logic, this is either active training or setting up few-shot prompts).

**5. Inference on Reddit Data**
Feeds the noisy, unstructured Reddit comments into the refined model. The model processes the "GenZ" slang and context to predict adverse events.

**6. Output Generation**
Saves the detected signals from the Reddit data into a structured format for analysis.

---

### Data Resources

**1. CADEC v2 (CSIRO Adverse Drug Event Corpus)**

* **Description:** A highly curated, expert-annotated dataset derived from medical forum posts. It contains text where specific spans are labeled as "Drug," "Adverse Effect," "Disease," etc.
* **Usage in Notebook:** This serves as the **"Textbook"** for the model. I use this data to teach (fine-tune) or validate the model, ensuring it understands the task of entity extraction based on gold-standard human annotations.
* **Link-to-data:**  https://data.csiro.au/collection/csiro:62387

**2. `raw_reddit_data.jsonl**`

* **Description:** My custom dataset containing raw, uncleaned posts scraped from 20 specific drug-related subreddits. It is full of slang, abbreviations, and informal language.
* **Usage in Notebook:** This serves as the **"Wild Environment."** I use this as the target for inference. Once the model understands the task via CADEC, it processes this file to detect previously unknown adverse drug reactions in real-world discussions.

## **1. Importing Packages**

Installing all the important packages and inporting the necessary libraries

In [2]:
!pip install transformers datasets seqeval nlpaug pytorch-crf scikit-learn nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 9.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=3e9d8cfca900a8ded648f4b22fd8c6d7c39fc98da0540cbe575a077a73e0fbcf
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
import os
import json
import glob
import random
import numpy as np
import pandas as pd
import torch
import nltk
from nltk.tokenize import wordpunct_tokenize
import zipfile
from transformers import AutoTokenizer, pipeline, DataCollatorForLanguageModeling, TrainingArguments, Trainer, AutoModelForMaskedLM, AutoModelForTokenClassification, DataCollatorForTokenClassification
from datasets import Dataset, concatenate_datasets
from huggingface_hub import notebook_login
import math
from sklearn.model_selection import train_test_split
import nlpaug.augmenter.char as nac
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

nltk.download('punkt')

print("Environment Setup Complete.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Environment Setup Complete.


## **2. Data - Processing**

### **2.1 - CADEC Data**

I sourced the **CADEC (CSIRO Adverse Drug Event Corpus)** dataset directly from the CSIRO Data Access Portal (Link given above). To prepare the environment, I downloaded the dataset, uploaded it to the runtime, and renamed the file to `cadec.zip` to ensure compatibility with my ingestion pipeline.

In this cell, I wrote a script to automate the extraction of the dataset, which is packaged as a complex nested archive. The code programmatically unpacks the layers and verifies the integrity of the directory structure. This ensures I have immediate access to the two critical components required for training:

* **`txt` Folder:** Contains the raw, unstructured patient narratives scraped from medical forums.
* **`original` Folder:** Contains the expert-verified annotations that map specific spans of text to medical entities, serving as the ground truth for my model.

In [5]:
# Step 1: Unzip the main 'cadec.zip'
print("📂 Unzipping the main 'cadec.zip'...")
with zipfile.ZipFile("cadec.zip", 'r') as zip_ref:
    zip_ref.extractall("temp_cadec_folder")

# Step 2: Find the nested 'data.zip' inside
data_zip_path = None
for root, dirs, files in os.walk("temp_cadec_folder"):
    if "data.zip" in files:
        data_zip_path = os.path.join(root, "data.zip")
        break

if data_zip_path:
    print(f"found nested zip at: {data_zip_path}")
    print("📂 Unzipping 'data.zip' to 'final_cadec_data'...")

    # Step 3: Unzip the inner data.zip
    with zipfile.ZipFile(data_zip_path, 'r') as zip_ref:
        zip_ref.extractall("final_cadec_data")

    print("✅ Done! Checking for folders...")

    # Step 4: Verify the folders are there
    base_path = "final_cadec_data"
    # Sometimes it extracts into a subfolder, so we check
    for root, dirs, files in os.walk("final_cadec_data"):
        if "txt" in dirs and "original" in dirs:
            base_path = root
            break

    print(f"\nSUCCESS. Data is ready at: {base_path}")
    print(f"   - Txt folder: {os.path.join(base_path, 'txt')}")
    print(f"   - Ann folder: {os.path.join(base_path, 'original')}") # .ann files are usually here
    print(f"   - Split folder: {os.path.join(os.path.dirname(base_path), 'split')}") # Split is usually one level up

else:
    print("❌ Error: Could not find 'data.zip' inside the extracted folder.")

📂 Unzipping the main 'cadec.zip'...
found nested zip at: temp_cadec_folder/data/data.zip
📂 Unzipping 'data.zip' to 'final_cadec_data'...
✅ Done! Checking for folders...

SUCCESS. Data is ready at: final_cadec_data
   - Txt folder: final_cadec_data/txt
   - Ann folder: final_cadec_data/original
   - Split folder: split


### **2.2 Parsing Cadec Data**

This is the core data processing step. The CADEC dataset provides raw text in one file and the "answers" (annotations) in a completely separate file using character indices (e.g., "the adverse event starts at character 50 and ends at character 60").

Models cannot learn from character indices; they need labels attached to specific words. This code aligns the two files to create a training format the model can understand.

**The Logic Flow**

1. **Path Correction**
I added a check at the beginning to handle directory inconsistencies. Depending on how the zip file extracted, the data might be in the root folder or nested inside a `data` subfolder. This logic automatically detects the correct path to prevent "File Not Found" errors.
2. **File Pairing**
I loop through every `.txt` file (the patient story) and look for its matching `.ann` file (the expert labels) by using the document ID. If a text file doesn't have a corresponding annotation file, I skip it to avoid breaking the dataset.
3. **Entity Extraction (Filtering for ADRs)**
The annotation files contain many types of tags (Drugs, Diseases, Findings), but for this project, I am strictly focused on side effects. I read the `.ann` file and filter only for lines containing the tag **'ADR'**. I store the start and end character positions of these side effects.
4. **Tokenization and Alignment (The Critical Step)**
This is the most complex part of the cell.
* I split the raw text sentences into individual words (tokens) using `wordpunct_tokenize`.
* I iterate through these words and track their exact character position in the original string.
* I check if that word's position falls inside one of the 'ADR' zones I saved earlier.


5. **BIO Labeling**
I assign a label to every single word using the standard BIO format:
* **O (Outside):** The word is not a side effect.
* **B-ADR (Beginning):** The word is the start of a side effect phrase (e.g., "severe" in "severe headache").
* **I-ADR (Inside):** The word is inside a side effect phrase (e.g., "headache" in "severe headache").



**The Result**
The function returns `parsed_data`, a clean list where every document is converted into two parallel lists: `tokens` (the words) and `ner_tags` (the labels). This is the exact format required to train a Named Entity Recognition (NER) model.

In [ ]:
# Ensure NLTK tokenizer is ready
nltk.download('punkt', quiet=True)

def parse_cadec(root_folder):
    # 1. FIX PATH: The data is one level deeper, inside 'data/data' or 'final_cadec_data/data'

    if os.path.exists(os.path.join(root_folder, "data", "txt")):
        base_path = os.path.join(root_folder, "data")
    else:
        base_path = root_folder

    print(f"📂 Looking for data in: {base_path}")

    txt_path_pattern = os.path.join(base_path, "txt", "*.txt")
    txt_files = glob.glob(txt_path_pattern)

    if not txt_files:
        print(f"Error: Still no .txt files found. Please check if {base_path}/txt exists.")
        return []

    print(f"Stats: Found {len(txt_files)} documents to parse...")
    parsed_data = []

    for txt_file in txt_files:
        doc_id = os.path.basename(txt_file).replace(".txt", "")
        ann_file = os.path.join(base_path, "ann", f"{doc_id}.ann")

        if not os.path.exists(ann_file):
            continue

        # Read Text
        with open(txt_file, 'r', encoding='utf-8') as f:
            text = f.read()

        # Read Annotations
        entities = []
        with open(ann_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('T') and 'ADR' in line:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        tag_info = parts[1].split()
                        tag_type = tag_info[0] # 'ADR'
                        start = int(tag_info[1])
                        end = int(tag_info[-1])
                        entities.append((start, end, tag_type))

        # Tokenize & Align
        tokens = wordpunct_tokenize(text)
        labels = []
        cursor = 0

        for token in tokens:
            start = text.find(token, cursor)
            end = start + len(token)
            cursor = end

            label = 'O'
            for es, ee, et in entities:
                if start >= es and end <= ee:
                    label = f"B-{et}" if start == es else f"I-{et}"
                    break
            labels.append(label)

        parsed_data.append({"id": doc_id, "tokens": tokens, "ner_tags": labels})

    return parsed_data

# Run the parser
cadec_data = parse_cadec("final_cadec_data")

print(f"✅ Successfully parsed {len(cadec_data)} documents.")
if len(cadec_data) > 0:
    print(f"Sample Tokens: {cadec_data[0]['tokens'][:10]}")
    print(f"Sample Tags:   {cadec_data[0]['ner_tags'][:10]}")

📂 Looking for data in: final_cadec_data/data
Stats: Found 3548 documents to parse...
✅ Successfully parsed 3548 documents.
Sample Tokens: ['Been', 'taking', 'Amlodopine', 'Besylate', 'three', 'years', 'now', 'and', 'it', 'lowered']
Sample Tags:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


### **2.3 Splitting the data**

I am dividing my parsed documents into three distinct groups: **Train** (to teach the model), **Dev** (to tune it), and **Test** (to grade it).

**The Logic**

1. **Checking for Official Rules**
The creators of the CADEC dataset actually provided a specific list of which documents should go into which group (`train.id`, `test.id`, etc.). I wrote this code to look for those files first. If I find them, I sort the data exactly as the researchers intended. This is crucial because it allows me to compare my results fairly against other scientific papers that used this same data.
2. **The "Plan B" (Random Split)**
If for some reason those official ID files are missing (maybe the zip extraction was weird), I included a fallback. I use a standard random splitter to shuffle the data and deal it out: 80% for training, 10% for validation, and 10% for testing.
3. **Sorting**
I loop through every document I prepared in the previous step, check its ID against the lists, and drop it into the correct bucket.

**The Result**
I end up with three separate piles of data. This ensures that when I test the model later, I am testing it on "unseen" data that it didn't memorize during training.

In [ ]:
def split_cadec_data(all_data, root_folder):
    # 1. Locate the split folder
    split_path = os.path.join(root_folder, "data", "split")

    if not os.path.exists(split_path):
        print(f"⚠️ Split folder not found at {split_path}. Doing a random split instead...")
        from sklearn.model_selection import train_test_split
        train, test = train_test_split(all_data, test_size=0.2, random_state=42)
        dev, test = train_test_split(test, test_size=0.5, random_state=42)
        return {'train': train, 'dev': dev, 'test': test}

    print(f"📂 Loading official splits from: {split_path}")

    # 2. Helper to load IDs from a file
    def load_ids(filename):
        path = os.path.join(split_path, filename)
        if os.path.exists(path):
            with open(path, 'r') as f:
                # Strip whitespace and remove empty lines
                return set(line.strip() for line in f if line.strip())
        return set()

    train_ids = load_ids("train.id")
    dev_ids = load_ids("dev.id")
    test_ids = load_ids("test.id")

    # 3. Sort data into buckets
    splits = {'train': [], 'dev': [], 'test': []}

    for doc in all_data:
        did = doc['id']
        if did in train_ids:
            splits['train'].append(doc)
        elif did in dev_ids:
            splits['dev'].append(doc)
        elif did in test_ids:
            splits['test'].append(doc)
        else:
            pass

    return splits

# Execute the split
cadec_splits = split_cadec_data(cadec_data, "final_cadec_data")

print(f"✅ Data Split Complete!")
print(f"   📘 Train: {len(cadec_splits['train'])} docs")
print(f"   Notebook Dev:   {len(cadec_splits['dev'])} docs")
print(f"   📕 Test:  {len(cadec_splits['test'])} docs")

📂 Loading official splits from: final_cadec_data/data/split
✅ Data Split Complete!
   📘 Train: 2485 docs
   Notebook Dev:   354 docs
   📕 Test:  709 docs


### **2.4 Raw reddit data**

I am ingesting the raw, real-world data that I want the model to analyze. Unlike the CADEC data, which was for training, this is the "wild" data where I want to detect new adverse drug reactions.

**The Data Source (`raw_reddit_data.jsonl`)**
This file contains thousands of raw user comments from drug-specific subreddits. I generated this dataset myself using a custom Python script named `ingest_reddit_history.py`. This script interacts with the **Pullpush API** to scrape historical Reddit data. You can find the source code for this scraper in the project's [GitHub repository here](https://github.com/Shreevatsa123/sentinel-pv).

**The Process**

1. **Loading the Tokenizer:** I initialize the BioBERT tokenizer ("The Professor"). This tool contains the dictionary required to translate medical words into numbers.
2. **Ingesting the File:** I read the JSONL file line-by-line. Since the scraper captures a lot of metadata (timestamps, upvotes, user IDs), I filter all that out and extract only the `text` field—the actual patient experience.
3. **Tokenization:** I convert these raw English sentences into numerical vectors. I chop the text into chunks of 128 tokens to ensure they fit uniformly into the model's memory. This prepares the social media chatter for mathematical processing.

In [ ]:
# 1. Load the "Professor" (Standard BioBERT Tokenizer)
model_checkpoint = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. Load your Raw Reddit Data
reddit_file = "raw_reddit_data.jsonl"
raw_texts = []

try:
    with open(reddit_file, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            # We only need the text content
            if 'text' in data:
                raw_texts.append(data['text'])

    print(f"✅ Loaded {len(raw_texts)} raw comments.")

except FileNotFoundError:
    print(f"❌ Error: Could not find '{reddit_file}'. Please upload it to the Files tab!")
    raw_texts = []

# 3. Process the Data (Tokenization)
# We convert text -> numbers. We chop them into chunks of 128 tokens.
if raw_texts:
    # Wrap in a Dataset object for speed
    raw_dataset = Dataset.from_dict({"text": raw_texts})

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

    print("⏳ Tokenizing data (this builds the bridge)...")
    tokenized_reddit_data = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

    print(f"✅ Data Ready for Adaptation!")
    print(f"   Sample Input: {tokenized_reddit_data[0]['input_ids'][:10]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Loaded 8171 raw comments.
⏳ Tokenizing data (this builds the bridge)...


Map:   0%|          | 0/8171 [00:00<?, ? examples/s]

✅ Data Ready for Adaptation!
   Sample Input: [101, 6243, 1128, 106, 178, 787, 1325, 5890, 1115, 1103]...


#### **Hugging Face Authentication**

This cell establishes a secure connection between my notebook environment and the Hugging Face Hub. When executed, it generates an interactive widget where I input my personal API token to authenticate the session.

**Why this is needed**
I am using **Llama 3.2**, which is a "gated" model. Unlike open-source models that anyone can download anonymously, Meta (the creators of Llama) requires users to accept a specific license agreement before using it. This login step provides the digital proof that I have accepted those terms and have been granted permission to download the model weights. Without this, the download in the next step would fail with a `403 Forbidden` error.

**How to create an Access Token**
If you need to replicate this step, here is how to generate your own token:

1. **Create an Account:** Go to [huggingface.co](https://huggingface.co/) and sign up.
2. **Accept Terms:** Search for "Llama 3.2" on the site, go to the model page, and click the button to accept the license agreement.
3. **Generate Token:**
* Click on your profile picture (top right)  **Settings**.
* Select **Access Tokens** from the menu on the left.
* Click **+ Create New Token**.
* Give it a name (e.g., "Colab_Project") and select **"Read"** permissions (since we are only downloading models).
* Copy the key starting with `hf_...` and paste it into the notebook widget.

In [ ]:
print("🔐 Hugging Face Login")
print("Paste your token in the box below and click 'Login'...")
notebook_login()

🔐 Hugging Face Login
Paste your token in the box below and click 'Login'...


## **3. Domain Adaptation (Creating "Street-Smart" BioBERT)**

This is the bridge between academic medicine and social media. The standard BioBERT model was trained on PubMed articles, so it understands "myocardial infarction" but has no idea what "heart feeling wonky" means.

In this cell, I am taking that academic model and forcing it to read my raw Reddit data. I am not teaching it *what* a side effect is yet; I am just teaching it how people talk on the internet so it doesn't get confused by slang later.

**The Logic**

1. **The "Masker" (Fill-in-the-Blank):**
I set up a process that randomly hides 15% of the words in the Reddit comments. The model has to guess the missing word based on the context. By doing this thousands of times, the model learns the patterns of "GenZ" medical language (e.g., learning that "finna" often precedes an action, or "brain zaps" is a sensation).
2. **Gentle Training:**
I set the learning rate very low (`2e-5`). This is crucial. I don't want to erase the model's existing medical knowledge; I just want to lightly tweak it to understand the new vocabulary. I call this "Domain Adaptation."
3. **Pushing to Cloud:**
Once the training is done (3 loops or "epochs" over the data), I save this new, specialized version of the model to my Hugging Face account (`sentinel-biobert-reddit-v2`). This allows me to pull this specific "street-smart" brain later for the actual detection tasks.

In [ ]:
# 1. The "Masker" (Data Collator)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# 2. Load the Model Architecture (Masked Language Model)
print("📥 Loading BioBERT for Adaptation...")
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

# 3. Training Settings
# I use a low learning rate (2e-5) to gently teach it new slang without breaking its medical knowledge.
training_args = TrainingArguments(
    output_dir="./biobert-v2-reddit-base",
    overwrite_output_dir=True,
    num_train_epochs=3,             # 3 passes is usually enough for adaptation
    per_device_train_batch_size=16, # T4 GPU handles this easily
    save_steps=500,
    save_total_limit=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                      # Fast training on GPU
    logging_steps=50,
    report_to="none",               # Keep output clean
    push_to_hub=True,
    hub_model_id="sentinel-biobert-reddit-v2"
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_reddit_data,
    data_collator=data_collator,
)

# 5. TRAIN! 🚀
print("🔥 STARTING DOMAIN ADAPTATION...")
trainer.train()

# 6. Save and Push
print("✅ Saving and Pushing to Hub...")
trainer.push_to_hub()
print("🎉 Success! Your 'Street Smart' model is now on the cloud.")

📥 Loading BioBERT for Adaptation...


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the model checkpoint at dmis-lab/biobert-base-cased-v1.2 were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


🔥 STARTING DOMAIN ADAPTATION...


Step,Training Loss
50,3.112000
100,2.822100
150,2.671400
200,2.624700
250,2.574900
300,2.479500
350,2.479700
400,2.460200
450,2.418300
500,2.376200


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...it-base/training_args.bin: 100%|##########| 5.84kB / 5.84kB            

  ...it-base/model.safetensors:   8%|7         | 33.5MB /  433MB            

🎉 Success! Your 'Street Smart' model is now on the cloud.


### **3.1: The Safe Training Loop**

I am cleaning up logical errors in the dataset labels before feeding them to the model. The BIO tagging format I generated earlier has strict rules, and raw data often violates them, which causes models (especially those with CRF layers) to crash or fail.

**The Logic**

1. **Format Safety Check:**
First, I check if my data is properly sorted into `train`, `dev`, and `test` groups. If something went wrong earlier and `cadec_data` is just one big list, I force a random split right here. This ensures the training code later finds the dictionary structure it expects.
2. **The "Orphan" Fix (The Core Task):**
In BIO tagging, specific sequences are mathematically impossible. You cannot be **"Inside" (I-ADR)** an entity unless you have **"Begun" (B-ADR)** it.
* **The Error:** `O`  `I-ADR` (You cannot be "inside" a side effect if you never started one).
* **The Fix:** I scan every single tag in the dataset. If I find an `I-ADR` tag that doesn't have a `B-ADR` immediately before it, I force it to become a `B-ADR`.



**Why I need this**
This ensures the data is "CRF-Safe." If I feed the model a sequence like "Outside -> Inside," it breaks the logical constraints of the algorithm. This script patches those holes so training runs smoothly.

In [ ]:
# 1. SAFETY CHECK: Ensure data is in the correct Dictionary format
if isinstance(cadec_data, list):
    print("⚠️ 'cadec_data' is a List. Converting to Train/Dev/Test splits now...")
    # Perform a random split if we lost the official ones
    train, test = train_test_split(cadec_data, test_size=0.2, random_state=42)
    dev, test = train_test_split(test, test_size=0.5, random_state=42)
    cadec_data = {'train': train, 'dev': dev, 'test': test}
    print("✅ Conversion Complete.")
else:
    print("✅ 'cadec_data' is already a Dictionary. Proceeding...")

# 2. DEFINE THE SANITIZER FUNCTION
def sanitize_data(split_data, split_name):
    fixed_count = 0
    total_tags = 0

    for doc in split_data:
        tags = doc['ner_tags']
        clean_tags = []

        # We assume the start of a document is always 'O' context
        prev_is_entity = False

        for i, tag in enumerate(tags):
            total_tags += 1

            if tag == 'I-ADR':
                if not prev_is_entity:
                    # 🚨 ILLEGAL MOVE: 'O' -> 'I-ADR'
                    # Fix: Change 'I-ADR' to 'B-ADR'
                    clean_tags.append('B-ADR')
                    fixed_count += 1
                    prev_is_entity = True
                else:
                    clean_tags.append(tag)
                    prev_is_entity = True

            elif tag == 'B-ADR':
                clean_tags.append(tag)
                prev_is_entity = True

            else: # Tag is 'O'
                clean_tags.append(tag)
                prev_is_entity = False

        # Update in place
        doc['ner_tags'] = clean_tags

    print(f"🧹 {split_name}: Scanned {total_tags} tags. Fixed {fixed_count} Orphans.")

# 3. RUN IT
print("🏥 Starting Data Sanitization...")
if 'train' in cadec_data:
    sanitize_data(cadec_data['train'], "Train")
    sanitize_data(cadec_data['dev'], "Dev")
    sanitize_data(cadec_data['test'], "Test")
else:
    print("❌ Critical Error: 'cadec_data' keys are missing. Please re-run the Data Acquisition cell.")

print("✅ Data is now CRF-Safe.")

⚠️ 'cadec_data' is a List. Converting to Train/Dev/Test splits now...
✅ Conversion Complete.
🏥 Starting Data Sanitization...
🧹 Train: Scanned 311751 tags. Fixed 33 Orphans.
🧹 Dev: Scanned 40171 tags. Fixed 6 Orphans.
🧹 Test: Scanned 34923 tags. Fixed 6 Orphans.
✅ Data is now CRF-Safe.


### **3.2 Data Tokenization and Alignment**

**What I am doing here**
I am converting my clean text data into the final numerical format for the model. This step is critical because I am now switching to the **custom tokenizer** I trained in the previous step (`Shreevatsa01/sentinel-biobert-reddit-v2`), which means the model now has a vocabulary adapted to social media slang.

**The Logic**

1. **Loading My Custom Tokenizer:**
I download the tokenizer associated with my "Street Smart" BioBERT. This ensures that when the model sees a word like "withdrawals," it processes it using the specific patterns it learned from the Reddit data.
2. **Creating the Translation Dictionary:**
The model deals in numbers, not strings. I scan my entire dataset to find every unique tag (like `B-ADR`, `I-ADR`, `O`) and assign each one a unique ID number (e.g., `B-ADR` = 1). This creates the map the model will use to output its predictions.
3. **The Alignment Problem (Crucial):**
This is the most complex part of this cell. Modern models like BERT break rare words into sub-word pieces (e.g., "Oxycontin" might become `["Oxy", "##con", "##tin"]`).
* **The Issue:** My original data has **one** label for "Oxycontin". But now the model sees **three** tokens. If I don't fix this, the lists won't match up, and the training will crash.
* **The Fix:** I wrote the `tokenize_and_align` function. It assigns the correct label to the first piece (`Oxy`) and assigns a special ignore code (`-100`) to the trailing pieces (`##con`, `##tin`). This tells the model: "Learn from the first piece, but don't get confused by the fragments."


4. **Final Conversion:**
I apply this logic to all my data splits (`train`, `dev`, `test`) and wrap them in the official Hugging Face `Dataset` format, which is optimized for fast GPU processing.

**The Result**
My text data is now fully digitized and aligned. Every word is broken down into tokens, and every token has a corresponding numeric label that accounts for sub-word splitting. The data is now ready for the final training loop.

In [ ]:
# --- STEP 5: PREPARE DATA FOR MODEL ---
from datasets import Dataset as HFDataset

MODEL_ID = "Shreevatsa01/sentinel-biobert-reddit-v2"
print(f"⏳ Downloading Tokenizer from: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 1. GENERATE LABEL MAPS (Tag -> Number)
unique_tags = set()
for split in cadec_data.keys():
    for doc in cadec_data[split]:
        unique_tags.update(doc['ner_tags'])

tag_list = sorted(list(unique_tags))
if 'O' in tag_list: tag_list.remove('O'); tag_list.insert(0, 'O')
label2id = {t: i for i, t in enumerate(tag_list)}
id2label = {i: t for t, i in label2id.items()}

print(f"✅ Labels Detected: {label2id}")

# 2. ALIGNMENT FUNCTION (Handles Sub-words)
def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens ([CLS], [SEP]) get -100
            elif word_idx != prev_word_idx:
                label_ids.append(label2id[label[word_idx]]) # Start of a new word
            else:
                label_ids.append(-100) # Sub-word parts (ignore them)
            prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

# 3. CONVERT TO HUGGING FACE DATASET
print("⏳ Converting to Hugging Face format...")
train_hf = HFDataset.from_list(cadec_data['train'])
dev_hf = HFDataset.from_list(cadec_data['dev'])
test_hf = HFDataset.from_list(cadec_data['test'])

# 4. APPLY TOKENIZATION
print("⏳ Aligning labels (this takes 10s)...")
tokenized_train = train_hf.map(tokenize_and_align, batched=True)
tokenized_dev = dev_hf.map(tokenize_and_align, batched=True)
tokenized_test = test_hf.map(tokenize_and_align, batched=True)

print(f"✅ Data Ready for Training!")
print(f"   Train Size: {len(tokenized_train)}")
print(f"   Dev Size:   {len(tokenized_dev)}")
print(f"   Test Size:  {len(tokenized_test)}")

⏳ Downloading Tokenizer from: Shreevatsa01/sentinel-biobert-reddit-v2...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

✅ Labels Detected: {'O': 0, 'B-ADR': 1, 'I-ADR': 2}
⏳ Converting to Hugging Face format...
⏳ Aligning labels (this takes 10s)...


Map:   0%|          | 0/2838 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

✅ Data Ready for Training!
   Train Size: 2838
   Dev Size:   355
   Test Size:  355


### **3.3 Data Augmentation**

Here I am deliberately adding errors to my training data to make the model tougher. Real users on Reddit do not proofread; they make typos, miss keys, and use shorthand. If I only train the model on perfect English, it will fail when it sees real-world messiness like "nausua" or "diziness."

**The Logic**

1. **The Typo Generator:**
I use a tool (`nac.KeyboardAug`) that understands the layout of a standard QWERTY keyboard. It doesn't just add random letters; it simulates "fat-finger" errors by swapping letters with their physical neighbors (e.g., hitting 'S' instead of 'A').
2. **Creating the "Broken" Data:**
I take my clean training set and create a copy. In this copy, I iterate through the words and apply a 30% chance to introduce a typo.
* **Crucial Detail:** I keep the **labels** exactly the same. If "headache" becomes "hezdache," I ensure it is still tagged as a Side Effect (`B-ADR`). This forces the model to learn that the concept of a side effect persists even if the spelling is wrong.


3. **Doubling the Dataset:**
I merge the original clean data with this new "broken" data. This effectively doubles my training size and gives the model two looks at every example: one perfect, and one noisy.
4. **Re-Tokenization:**
Because I changed the actual words (creating new sub-words), I have to run the tokenizer again on this combined list to generate the correct numerical inputs for the model.

**The Result**
I now have a training set that mimics the chaos of the internet. The model will learn to look at the *context* around a word rather than relying solely on perfect spelling.

In [ ]:
print("🧪 Initializing Typo Generator (Keyboard Noise)...")
# Simulates typing errors (e.g. hitting 's' instead of 'a')
aug = nac.KeyboardAug(aug_char_p=0.3, aug_word_p=0.3, include_special_char=False)

def augment_data(example):
    original_tokens = example['tokens']
    new_tokens = []

    # We iterate token by token to ensure labels stay aligned
    for token in original_tokens:
        # 30% chance to typo a word
        if len(token) > 3 and random.random() < 0.3:
            try:
                # aug.augment returns a list e.g. ['typo']
                augmented_text = aug.augment(token)
                # Handle nlpaug return types (sometimes string, sometimes list)
                if isinstance(augmented_text, list):
                    new_tokens.append(augmented_text[0])
                else:
                    new_tokens.append(augmented_text)
            except:
                # If augmentation fails, keep original
                new_tokens.append(token)
        else:
            new_tokens.append(token)

    return {
        'tokens': new_tokens,
        'ner_tags': example['ner_tags'], # Labels remain exactly the same
        'id': f"{example['id']}_aug"
    }

# 1. Select a subset to augment (e.g., 50% of training data)
print(f"   Original Train Size: {len(train_hf)}")
print("   🔨 Generating 'Broken' Reddit-style examples...")
augmented_dataset = train_hf.map(augment_data)

# 2. Combine Original + Broken Data
combined_train = concatenate_datasets([train_hf, augmented_dataset])
print(f"✅ Robustness Injection Complete.")
print(f"   New Train Size: {len(combined_train)} samples")

# 3. RE-TOKENIZE
# We must re-tokenize this new larger dataset
print("⏳ Aligning labels for the new augmented dataset...")
tokenized_train_final = combined_train.map(tokenize_and_align, batched=True)
print("✅ Ready for Training.")

🧪 Initializing Typo Generator (Keyboard Noise)...
   Original Train Size: 2838
   🔨 Generating 'Broken' Reddit-style examples...


Map:   0%|          | 0/2838 [00:00<?, ? examples/s]

✅ Robustness Injection Complete.
   New Train Size: 5676 samples
⏳ Aligning labels for the new augmented dataset...


Map:   0%|          | 0/5676 [00:00<?, ? examples/s]

✅ Ready for Training.


## **4. Final Model Training**

This is the moment where I actually build the final product. Up until now, I taught the model *slang* and prepared the *data*. Now, I am forcing the model to use that slang knowledge to solve the specific problem: finding adverse drug reactions.

**The Logic**

1. **Switching the "Brain" Structure:**
I load my "Street Smart" model (`sentinel-biobert-reddit-v2`), but I change its job description.
* Before, it was a `MaskedLM` (Fill-in-the-Blank).
* Now, I load it as `AutoModelForTokenClassification`. This tells the model: "Stop guessing missing words. Start assigning a label (Side Effect vs. Not Side Effect) to every single word you see."


2. **Setting the Rules (`TrainingArguments`):**
* **Low Learning Rate (`2e-5`):** I keep this number tiny. If I make it too high, the model will panic and forget all the Reddit slang I just taught it. I want it to *gently* learn the new task while keeping its street smarts.
* **Evaluation Strategy:** I tell the model to take a practice test after every "epoch" (full pass through the data) so I can see if it's actually getting smarter.


3. **The "Grading Rubric" (`compute_metrics`):**
Models are messy. They output predictions for everything, including those weird sub-word pieces (`##ing`, `##ed`) and padding tokens.
* **The Filter:** I wrote code here to strictly **ignore** anything labeled `-100`. I only grade the model on whole, real words.
* **The Score:** I use the **F1 Score** to grade it. This is better than just "accuracy" because it balances precision (not false alarming) and recall (not missing real side effects).


4. **The "Fair Exam" Setup:**
In the `Trainer`, I make a very specific choice:
* **Train on:** `tokenized_train_final` (The **Augmented/Messy** data with typos).
* **Evaluate on:** `tokenized_dev` (The **Clean** data).
* **Why:** This is like practicing for a test in a noisy coffee shop so that when you take the real test in a quiet room, it feels easy. I train it on hard mode so it performs perfectly on normal mode.



**The Result**
When this finishes, "Sentinel V3" is born. It is now a model that understands Reddit slang *and* knows how to medically tag symptoms with high accuracy.


In [ ]:
# 1. LOAD MODEL
print(f"🚀 Loading architecture for: {MODEL_ID}")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. TRAINING ARGUMENTS (Updated for new Transformers version)
args = TrainingArguments(
    output_dir="./sentinel-v3-final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,             # Low LR to preserve Reddit knowledge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 3. METRICS FUNCTION
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [tag_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [tag_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_predictions),
        "precision": 0.0,
        "recall": 0.0
    }

# 4. INITIALIZE TRAINER
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_final, # <--- Uses the AUGMENTED data
    eval_dataset=tokenized_dev,          # Validate on clean data (The "Fair Exam")
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

# 5. RUN TRAINING
print("🔥 STARTING FINAL TRAINING (With Typo Augmentation)...")
trainer.train()

print("🎉 DONE! Sentinel V3 is built.")

🚀 Loading architecture for: Shreevatsa01/sentinel-biobert-reddit-v2


Some weights of BertForTokenClassification were not initialized from the model checkpoint at Shreevatsa01/sentinel-biobert-reddit-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-993057886.py:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🔥 STARTING FINAL TRAINING (With Typo Augmentation)...


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.156100,0.161709,0.740224,0.000000,0.000000
2,0.123300,0.165343,0.745577,0.000000,0.000000
3,0.097000,0.185051,0.757767,0.000000,0.000000
4,0.081000,0.200434,0.749627,0.000000,0.000000


🎉 DONE! Sentinel V3 is built.


### **4.1 Evaluation**

I am manually triggering a final evaluation to get the true performance numbers. The default "accuracy" metric is useless here because most words in a sentence are *not* side effects. A model that ignores everything would still look 90% accurate but be 100% useless.

**The Logic**

1. **The "Fixed" Rubric:**
I replaced the standard scoring function with one that calculates three specific numbers:
* **Precision:** When the model claims something is a side effect, how often is it right? (Low precision = Crying Wolf).
* **Recall:** Out of all the real side effects in the text, how many did the model actually find? (Low recall = Missing danger signals).
* **F1 Score:** The balance between the two.


2. **Filtering Noise:**
Just like in the training step, I ensure the code ignores the `-100` tags (sub-word pieces and padding). I only want to grade the model on its ability to label whole words correctly.
3. **Immediate Execution:**
I attach this new grading logic to the `trainer` and force it to run a standalone test right now.

**The Result**
This outputs the "Real Metrics Report." These numbers tell me if the model is actually ready to be trusted with real patient data or if it needs more training.

In [ ]:
# 1. Define the CORRECTED metrics function
def compute_metrics_fixed(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [tag_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [tag_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_predictions),
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions)
    }

# 2. Update the Trainer with the new function
trainer.compute_metrics = compute_metrics_fixed

# 3. Run evaluation immediately (takes 10 seconds)
print("📊 Calculating Real Precision & Recall...")
metrics = trainer.evaluate()

print("\n🏥 REAL METRICS REPORT:")
print(f"   ✅ F1 Score:  {metrics['eval_f1']:.4f}")
print(f"   🎯 Precision: {metrics['eval_precision']:.4f}")
print(f"   👀 Recall:    {metrics['eval_recall']:.4f}")

📊 Calculating Real Precision & Recall...



🏥 REAL METRICS REPORT:
   ✅ F1 Score:  0.7578
   🎯 Precision: 0.7393
   👀 Recall:    0.7772


## **5. Model Deployment**

I am taking the fully trained model and uploading it to my Hugging Face profile.

**The Logic**

1. **Pushing the Model:**
I send the actual neural network (the weights it adjusted during training) to the cloud repository named `sentinel-v3-final`.
2. **Pushing the Tokenizer (Critical):**
I also upload the tokenizer. This is the specific dictionary that translates "GenZ" slang into the numbers my model understands. If I forget this, the model is useless because nobody else has a dictionary that includes my specific Reddit vocabulary.

**The Result**
The model is now permanently stored online. I can now delete this notebook, open a completely new application (like a web app), and download this exact brain using the link provided.

In [ ]:
# --- STEP 9: SAVE TO HUGGING FACE ---

repo_name = "sentinel-v3-final"

print(f"☁️ Uploading {repo_name} to Hugging Face...")

# 1. Push the Model & Training Stats
trainer.push_to_hub(repo_name)

# 2. Push the Tokenizer (Critical for inference later)
tokenizer.push_to_hub(repo_name)

print(f"🎉 SUCCESS! Your model is live at: https://huggingface.co/Shreevatsa01/{repo_name}")

☁️ Uploading sentinel-v3-final to Hugging Face...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3-final/model.safetensors:   0%|          |  558kB /  431MB            

  ...3-final/training_args.bin:   2%|1         |   112B / 5.84kB            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


🎉 SUCCESS! Your model is live at: https://huggingface.co/Shreevatsa01/sentinel-v3-final


## **6. Inference Testing (Sanity Check)**

I am running a live test on the model I just trained. Before I trust this model with millions of Reddit comments, I need to see if it actually works on a few tricky examples right here in the notebook.

**The Logic**

1. **Freezing the Brain (`model.eval()`):**
I command the model to stop learning. This turns off features like "Dropout" that add randomness during training. I want consistent, reliable answers now, not creative guessing.
2. **Building the Pipeline:**
I use Hugging Face's `pipeline` tool to wrap my complex model into a simple function.
* **Crucial Detail (`aggregation_strategy="simple"`):** Remember how the model breaks words into pieces (like `head`, `##ache`)? This setting tells the pipeline to glue them back together automatically. So instead of getting two separate results, I just get one clean result: "headache".


3. **The Stress Test:**
I manually wrote four specific sentences to test the model's limits:
* **Typos:** "headaache" (Can it handle bad spelling?)
* **Description:** "burning" (Can it understand sensations, not just medical terms?)
* **Slang:** "weird tummy" (Can it handle the GenZ talk I trained it on?)


4. **Confidence Filter:**
I set a rule to only show me results where the model is more than **50% sure**. If the model is guessing, I don't want to hear about it.

**The Result**
This prints a "Diagnosis Report." It proves immediately if my training worked by showing me exactly which symptoms the model spotted in my test sentences.

In [ ]:
model.eval()

print("🚀 Using the live model from memory...")
device = 0 if torch.cuda.is_available() else -1

# We pass the 'model' object, not the folder path string
nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device
)

# 2. Define Test Sentences
test_sentences = [
    "I took 50mg of zoloft and now I have a terrible headaache and nausea.",  # Typo
    "My stomach feels like its burning after taking the pill.",               # Descriptive
    "started ozempic yesterday, feeling super dizzy and weird tummy.",        # Slang
    "no side effects so far, just a bit of tiredness."                        # Mild
]

# 3. Run Predictions
print("\n🏥 SENTINEL V3 DIAGNOSIS REPORT:")
print("-" * 60)

for text in test_sentences:
    results = nlp(text)
    print(f"📝 Text: \"{text}\"")
    if not results:
        print("   ✅ No Symptoms Detected.")
    else:
        for entity in results:
            # High confidence filter
            if entity['score'] > 0.50:
                print(f"   ⚠️ SYMPTOM DETECTED: {entity['word']} (Confidence: {entity['score']:.2f})")
    print("-" * 60)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🚀 Using the live model from memory...

🏥 SENTINEL V3 DIAGNOSIS REPORT:
------------------------------------------------------------
📝 Text: "I took 50mg of zoloft and now I have a terrible headaache and nausea."
   ⚠️ SYMPTOM DETECTED: headaache (Confidence: 0.96)
   ⚠️ SYMPTOM DETECTED: nausea (Confidence: 0.98)
------------------------------------------------------------
📝 Text: "My stomach feels like its burning after taking the pill."
   ⚠️ SYMPTOM DETECTED: stomach feels like its burning (Confidence: 0.84)
------------------------------------------------------------
📝 Text: "started ozempic yesterday, feeling super dizzy and weird tummy."
   ⚠️ SYMPTOM DETECTED: dizzy (Confidence: 0.64)
   ⚠️ SYMPTOM DETECTED: weird tummy (Confidence: 0.93)
------------------------------------------------------------
📝 Text: "no side effects so far, just a bit of tiredness."
   ⚠️ SYMPTOM DETECTED: tiredness (Confidence: 0.92)
------------------------------------------------------------


## **7. Cloud Verification (The "Stranger Test")**


In this final step, I am performing a "Stranger Test" to verify my deployment. Instead of using the model currently sitting in my memory, I force the system to download it fresh from the Hugging Face cloud, just like an external user or web app would. By successfully running a quick test sentence on this downloaded version, I confirm that the upload worked perfectly and the model is live, accessible, and ready for real-world use outside this notebook.

In [ ]:
model_id = "Shreevatsa01/sentinel-v3-final"

print(f"🌍 Downloading model from Hugging Face: {model_id}...")

# 2. Initialize Pipeline
try:
    nlp = pipeline("token-classification", model=model_id, aggregation_strategy="simple")
    print("✅ Download Successful! The model is live on the cloud.")

    # 3. Quick Test
    text = "I have a weird headaache and tummy pain."
    results = nlp(text)
    print(f"\n📝 Test Input: '{text}'")
    for r in results:
        print(f"   ⚠️ Found: {r['word']} ({r['score']:.2f})")

except Exception as e:
    print(f"❌ Error: {e}")
    print("Check your Hugging Face repo URL to ensure the files are there.")

🌍 Downloading model from Hugging Face: Shreevatsa01/sentinel-v3-final...


config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


✅ Download Successful! The model is live on the cloud.

📝 Test Input: 'I have a weird headaache and tummy pain.'
   ⚠️ Found: headaache (0.97)
   ⚠️ Found: tummy pain (0.83)
